In [44]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [45]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
    [results_1l, results_2l,],
    ignore_index=True )
#results = 

In [46]:
results.to_excel("resultados.xlsx")

In [47]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,MSE_ZZx1_theta,R2_ZZx2_theta,MSE_ZZx2_theta,...,R2_LSG_1_theta,MSE_LSG_1_theta,R2_LSG_2_theta,MSE_LSG_2_theta,R2_ZZx1_inv_theta,MSE_ZZx1_inv_theta,R2_zzx2_inv2_theta,MSE_zzx2_inv2_theta,R2_semiCirc_theta,MSE_semiCirc_theta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed4705,[1],0.3,0.7,0.01,4705,-0.179891,-0.000075,0.023074,-0.000823,...,-11.781966,-0.055668,-0.748364,-0.010603,-1.596818,-0.026972,-0.962770,-0.002709,-0.060649,-0.009015
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed2693,[1],0.3,0.7,0.01,2693,-0.214286,0.006568,0.015715,0.000107,...,-11.893951,-0.050805,-0.816748,-0.005875,-1.685503,-0.018900,-1.022080,-0.000460,-0.071675,-0.007226
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed4649,[1],0.3,0.7,0.01,4649,-0.210807,0.000878,0.012015,-0.000106,...,-11.975781,-0.056081,-0.808379,-0.010579,-1.689256,-0.027179,-1.022090,-0.002587,-0.077336,-0.009393
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed3633,[1],0.3,0.7,0.01,3633,0.761277,0.677203,-0.823908,0.462480,...,0.854605,0.595732,-4.782161,0.319042,0.018808,0.449090,-7.153187,0.079361,-30.210797,-0.196762
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed7789,[1],0.3,0.7,0.01,7789,-0.205858,0.002878,0.013170,0.001330,...,-11.954769,-0.054811,-0.796903,-0.009480,-1.683535,-0.025292,-1.038555,-0.001855,-0.078327,-0.010961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3086,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed6274,"[3, 1, 1]",0.7,0.3,0.90,6274,0.547843,0.582458,0.787891,0.362234,...,-2.400522,0.442405,-2.617589,0.255157,-2.139094,0.276627,-8.083965,0.142178,-10.509687,0.094532
3087,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed7805,"[3, 1, 1]",0.7,0.3,0.90,7805,0.524860,0.498866,0.635631,0.338082,...,-4.283150,0.347733,-2.038816,0.197777,-1.616691,0.271634,-5.912504,0.183614,-5.586486,0.102204
3088,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed3109,"[3, 1, 1]",0.7,0.3,0.90,3109,0.561345,0.517563,0.620186,0.353962,...,-3.522649,0.374510,-2.326809,0.224695,-0.493140,0.294403,-4.654585,0.193294,-7.037452,0.096106
3089,model_arch3-1-1_r0.9_Ld0.7_Lp0.3_seed8496,"[3, 1, 1]",0.7,0.3,0.90,8496,0.560286,0.572841,-0.306579,0.350755,...,-2.391176,0.415491,-2.747356,0.223955,-0.831566,0.334386,-6.818385,0.161922,-8.428751,0.083885


In [48]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
   # "ZZy1":     "Test",
   # "ZZy2":     "Test",
    "LSG-1":    "Test",
    #"LSG-2":    "Test",
    #"ZZx1-inv": "Test",
    #"ZZx2-inv2": "Test",
   # "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] -
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
2435,model_arch44_r0.9_Ld0.5_Lp0.5_seed4358,[44],0.927520,0.928277,0.876637,0.896183
2199,model_arch20_r0.9_Ld0.5_Lp0.5_seed1257,[20],0.902600,0.953277,0.873891,0.895776
2357,model_arch36_r0.9_Ld0.5_Lp0.5_seed686,[36],0.914859,0.934707,0.865683,0.892379
751,model_arch38_r0.01_Ld0.7_Lp0.3_seed2693,[38],0.946669,0.902090,0.868311,0.890775
2262,model_arch27_r0.01_Ld0.5_Lp0.5_seed686,[27],0.905555,0.963866,0.851514,0.890375



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZxReto_theta,R2_LSG_1_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
2435,model_arch44_r0.9_Ld0.5_Lp0.5_seed4358,[44],0.927520,0.928277,0.933718,0.819557,0.927520,0.928277,0.876637,0.896183
2199,model_arch20_r0.9_Ld0.5_Lp0.5_seed1257,[20],0.902600,0.953277,0.915351,0.832431,0.902600,0.953277,0.873891,0.895776
2357,model_arch36_r0.9_Ld0.5_Lp0.5_seed686,[36],0.914859,0.934707,0.853153,0.878214,0.914859,0.934707,0.865683,0.892379
751,model_arch38_r0.01_Ld0.7_Lp0.3_seed2693,[38],0.946669,0.902090,0.923804,0.812819,0.946669,0.902090,0.868311,0.890775
2262,model_arch27_r0.01_Ld0.5_Lp0.5_seed686,[27],0.905555,0.963866,0.916344,0.786684,0.905555,0.963866,0.851514,0.890375


In [49]:
final_table.to_excel("BestModels-3tst.xlsx")

In [51]:
# ============================================
# MÉDIA, DESVIO, MÍNIMO E MÁXIMO
# ============================================
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
   "ZZy1":     "Test",
   "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv2": "Test",
   "semiCirc": "Test",
}
summary_tables = {}

for target in TARGETS:

    top_df = results.copy()
    rows = []

    for s in SETS_CATEGORY.keys():

        r2_col = f"R2_{s.replace('-', '_')}_{target}"
        mse_col = f"MSE_{s.replace('-', '_')}_{target}"

        row = {
            "Set": s,
            "Category": SETS_CATEGORY[s]
        }

        # =========================
        # R²
        # =========================
        if r2_col in top_df.columns:
            row["R2_mean"] = top_df[r2_col].mean()
            row["R2_std"]  = top_df[r2_col].std()
            row["R2_min"]  = top_df[r2_col].min()
            row["R2_max"]  = top_df[r2_col].max()
        else:
            row["R2_mean"] = np.nan
            row["R2_std"]  = np.nan
            row["R2_min"]  = np.nan
            row["R2_max"]  = np.nan

        # =========================
        # MSE
        # =========================
        if mse_col in top_df.columns:
            row["MSE_mean"] = top_df[mse_col].mean()
            row["MSE_std"]  = top_df[mse_col].std()
            row["MSE_min"]  = top_df[mse_col].min()
            row["MSE_max"]  = top_df[mse_col].max()
        else:
            row["MSE_mean"] = np.nan
            row["MSE_std"]  = np.nan
            row["MSE_min"]  = np.nan
            row["MSE_max"]  = np.nan

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    summary_tables[target] = summary_df

    # =========================
    # MOSTRA SOMENTE A TABELA
    # =========================
    display(
        summary_df.style.format({
            "R2_mean": "{:.4f}",
            "R2_std":  "{:.4f}",
            "R2_min":  "{:.4f}",
            "R2_max":  "{:.4f}",

            "MSE_mean": "{:.6f}",
            "MSE_std":  "{:.6f}",
            "MSE_min":  "{:.6f}",
            "MSE_max":  "{:.6f}"
        })
    )

,Set,Category,R2_mean,R2_std,R2_min,R2_max,MSE_mean,MSE_std,MSE_min,MSE_max
0,ZZx1,Train,0.7710,0.1614,-0.2143,0.9754,0.670149,0.112396,-0.000075,0.867753
1,ZZx2,Val,0.5118,0.3526,-1.7426,0.9798,0.496663,0.050801,-0.000823,0.580696
2,ZZxReto,Test,0.5994,0.3591,-1.3965,0.9530,0.634758,0.067576,-0.003929,0.723740
3,ZZy1,Test,-13.7118,4.8586,-29.6710,0.0900,0.267295,0.051331,-0.000283,0.356323
4,ZZy2,Test,-13.4927,3.5403,-28.3890,-4.2146,0.097251,0.253125,-0.967545,0.422680
5,LSG-1,Test,0.1485,1.5847,-12.4621,0.8953,0.557948,0.070353,-0.056420,0.662317
6,LSG-2,Test,-2.7365,1.1107,-8.4001,-0.0611,0.326806,0.043920,-0.010603,0.426456
7,ZZx1-inv,Test,-1.4837,1.7076,-10.2589,0.6358,0.479469,0.059711,-0.027179,0.613952
8,ZZx2-inv2,Test,nan,nan,nan,nan,nan,nan,nan,nan
9,semiCirc,Test,-24.5583,7.5633,-52.3284,-0.0606,-0.117321,0.121549,-0.725386,0.161209
